# FLOW — Vocabulary Growth (Corpus-Grounded)

**Geometric Causal Architecture** — weight-free, token-free reasoning.  
Knowledge stored as shape in a 104D Riemannian manifold.

This notebook **loads the K-12 model** as a base and feeds **5 real-world corpora**  
(2,000 texts each = 10,000 total) through the VocabularyBuilder PMI pipeline  
to grow a **semantically grounded** vocabulary.

### Why this matters

The K-12 model places words by character n-grams (spelling similarity).  
This gives NO semantic structure — "cat" and "car" end up near each other  
because they share letters, not meaning.

**PMI co-occurrence from real text** fixes this: words that appear together  
in actual sentences get pulled together on the manifold. "cat" moves near  
"kitten" and "pet", while "car" moves near "drive" and "road".

### Corpus mix

| Dataset | Texts | Domain |
|---|---|---|
| Simple English Wikipedia | 2,000 | General knowledge |
| OpenAssistant (oasst1) | 2,000 | Agent dialogue |
| Alpaca (cleaned) | 2,000 | Instruction following |
| Glaive Function Calling v2 | 2,000 | Tool calling |
| SlimOrca | 2,000 | Reasoning chains |

**Hardware:** Kaggle CPU (4 cores, 30 GB RAM)  
**Estimated runtime:** 10–20 minutes  
**Base model:** K-12 pipeline (`FLOW-K12_manifold.npz` + `FLOW-K12_vocab.npz`)  
**Output:** `FLOW-Grounded_manifold.npz` + `FLOW-Grounded_vocab.npz`

## 1. Setup — Clone/refresh repo & install dependencies

In [ ]:
import os, subprocess

# ── GitHub PAT from Kaggle Secrets ──────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    PAT = UserSecretsClient().get_secret("GITHUB_PAT")
    print(f"✓ PAT loaded from Kaggle Secrets ({len(PAT)} chars)")
except Exception:
    PAT = None
    print("⚠ No Kaggle Secrets — using public clone (push disabled)")

REPO_URL = f"https://{PAT}@github.com/Unseengap/FLOW.git" if PAT else "https://github.com/Unseengap/FLOW.git"
REPO_DIR = "FLOW"

# ── Clone or refresh ──────────────────────────────────────────
if os.path.isdir(REPO_DIR):
    print("Existing FLOW clone detected — refreshing...")
    before = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    # Update remote URL (in case PAT changed)
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "fetch", "origin", "main"],
                   cwd=REPO_DIR, capture_output=True, text=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"],
                   cwd=REPO_DIR, capture_output=True, text=True)
    after = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    if before == after:
        print(f"  Already up-to-date ({after})")
    else:
        log = subprocess.run(
            ["git", "--no-pager", "log", "--oneline", f"{before}..{after}"],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        print(f"  Updated {before} → {after}")
        for line in log.stdout.strip().split("\n")[:10]:
            if line.strip():
                print(f"    {line}")
else:
    print("Cloning FLOW repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR],
                   capture_output=True, text=True, check=True)
    after = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    print(f"  ✓ Cloned at {after}")

assert os.path.isdir("FLOW/src"), "FLOW repo not found"
print(f"✓ Repository ready")

In [ ]:
# Install dependencies (pure math — no ML frameworks)
!pip install numpy scipy networkx datasets -q

# Optional acceleration
!pip install faiss-cpu -q 2>/dev/null || echo "FAISS not available — linear scan fallback"

print("✓ Dependencies installed")

In [ ]:
# Add FLOW to Python path
import sys
sys.path.insert(0, "FLOW")

# Core imports
from src.phase5 import GEOPipeline, PipelineResult
from src.vocabulary import VocabularyBuilder, VocabularyStore
from src.persistence import ManifoldSnapshot
from src.phase3.annealing_engine.experience import Experience
from src.phase1.expression.matcher import ExpressionEntry
import numpy as np
import time

print("✓ All FLOW imports successful")
print(f"  numpy {np.__version__}")

## 2. Load K-12 Base Model

Start from the K-12 pipeline model which has curriculum concepts  
from Kindergarten through Grade 12. Falls back to the V1 base model  
if K-12 is not available.

In [ ]:
# ── Locate best available base model ───────────────────────
MODELS_DIR = f"{REPO_DIR}/models"

# Prefer K-12 model, fall back to V1 base
if os.path.isfile(f"{MODELS_DIR}/FLOW-K12_manifold.npz"):
    BASE_MANIFOLD = f"{MODELS_DIR}/FLOW-K12_manifold.npz"
    BASE_VOCAB    = f"{MODELS_DIR}/FLOW-K12_vocab.npz"
    base_name = "K-12 pipeline"
elif os.path.isfile(f"{MODELS_DIR}/FLOW-V1-Base_manifold.npz"):
    BASE_MANIFOLD = f"{MODELS_DIR}/FLOW-V1-Base_manifold.npz"
    BASE_VOCAB    = f"{MODELS_DIR}/FLOW-V1-Base_vocab.npz"
    base_name = "V1 base"
else:
    raise FileNotFoundError(
        f"No model found in {MODELS_DIR}/. "
        "Run kaggle_base_model.ipynb or kaggle_k12_pipeline.ipynb first."
    )

print(f"Loading {base_name} model...")
print(f"  Manifold : {BASE_MANIFOLD}")
print(f"  Vocab    : {BASE_VOCAB}")

# ── Load pipeline ─────────────────────────────────────────
# T0=1.0 : high initial temperature for flexible placement
# lambda_=0.005 : slower cooling for large-scale ingestion
# T_floor=0.03  : lower floor to allow crystallisation over time
pipeline = GEOPipeline.load(
    BASE_MANIFOLD,
    vocabulary_path=BASE_VOCAB,
    T0=1.0,
    lambda_=0.005,
    T_floor=0.03,
    flow_seed=42,
)

# Record base vocabulary for later merging
base_vocab_entries = VocabularyStore.load(BASE_VOCAB)
n_base_vocab = len(base_vocab_entries)

print(f"\n✓ {base_name} model loaded")
print(f"  Manifold dimension : {pipeline.manifold.DIM}")
print(f"  Base concepts      : {pipeline.n_concepts:,}")
print(f"  Base vocab entries : {n_base_vocab:,}")
print(f"  Temperature        : {pipeline.temperature:.4f}")

## 3. Load Corpora — 5 datasets × 2,000 texts

Five complementary datasets that ground FLOW in real language:
- **Wikipedia** — broad concept coverage, clean prose
- **OpenAssistant** — conversational Q&A patterns
- **Alpaca** — imperative/procedural instructions
- **Glaive Functions** — structured tool-calling language
- **SlimOrca** — step-by-step reasoning chains

In [ ]:
from datasets import load_dataset
from collections import Counter

all_texts = []   # collect (text, source_tag) pairs

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
N_PER_DATASET = 2_000     # 5 datasets × 2K = 10K total texts
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 3A. Wikipedia — Simple English (general knowledge)
print("Loading Simple English Wikipedia...")
ds_wiki = load_dataset("wikimedia/wikipedia", "20231101.simple",
                       split="train", trust_remote_code=True)
for i, article in enumerate(ds_wiki):
    if i >= N_PER_DATASET:
        break
    text = article.get("text", "")
    if text and len(text) > 50:
        all_texts.append((text, "wikipedia"))
print(f"  ✓ Wikipedia: {len([t for t in all_texts if t[1]=='wikipedia']):,}")

# 3B. OpenAssistant (oasst1) — dialogue
print("Loading OpenAssistant (oasst1)...")
ds_oasst = load_dataset("OpenAssistant/oasst1", split="train",
                        trust_remote_code=True)
for i, row in enumerate(ds_oasst):
    if i >= N_PER_DATASET:
        break
    text = row.get("text", "")
    if text and len(text) > 30:
        all_texts.append((text, "openassistant"))
print(f"  ✓ OpenAssistant: {len([t for t in all_texts if t[1]=='openassistant']):,}")

# 3C. Alpaca (cleaned) — instruction following
print("Loading Alpaca (cleaned)...")
ds_alpaca = load_dataset("yahma/alpaca-cleaned", split="train",
                         trust_remote_code=True)
for i, row in enumerate(ds_alpaca):
    if i >= N_PER_DATASET:
        break
    parts = []
    if row.get("instruction"): parts.append(row["instruction"])
    if row.get("input"): parts.append(row["input"])
    if row.get("output"): parts.append(row["output"])
    text = " ".join(parts)
    if text and len(text) > 30:
        all_texts.append((text, "alpaca"))
print(f"  ✓ Alpaca: {len([t for t in all_texts if t[1]=='alpaca']):,}")

# 3D. Glaive Function Calling v2 — tool calling
print("Loading Glaive Function Calling v2...")
ds_glaive = load_dataset("glaiveai/glaive-function-calling-v2",
                         split="train", trust_remote_code=True)
for i, row in enumerate(ds_glaive):
    if i >= N_PER_DATASET:
        break
    text_parts = []
    if row.get("system"): text_parts.append(row["system"])
    if row.get("chat"): text_parts.append(row["chat"])
    text = " ".join(text_parts)
    if text and len(text) > 30:
        all_texts.append((text, "glaive_functions"))
print(f"  ✓ Glaive Functions: {len([t for t in all_texts if t[1]=='glaive_functions']):,}")

# 3E. SlimOrca — reasoning chains
print("Loading SlimOrca...")
ds_orca = load_dataset("Open-Orca/SlimOrca", split="train",
                       trust_remote_code=True)
for i, row in enumerate(ds_orca):
    if i >= N_PER_DATASET:
        break
    convos = row.get("conversations", [])
    text = " ".join(
        turn.get("value", "") for turn in convos if isinstance(turn, dict)
    )
    if text and len(text) > 30:
        all_texts.append((text, "slimorca"))
print(f"  ✓ SlimOrca: {len([t for t in all_texts if t[1]=='slimorca']):,}")

# Summary
source_counts = Counter(t[1] for t in all_texts)
print(f"\n{'='*60}")
print(f"Total texts loaded: {len(all_texts):,}")
for src, cnt in source_counts.most_common():
    print(f"  {src:20s} : {cnt:>6,}")
print(f"{'='*60}")

## 4. Configure VocabularyBuilder & Feed Corpora

In [ ]:
# ── Configuration ─────────────────────────────────────────────
V_MAX             = 10_000   # vocabulary ceiling
MIN_COUNT         = 3        # lower threshold for 10K corpus
WINDOW_SIZE       = 5        # co-occurrence window
N_CONTRAST_PASSES = 2        # C4 refinement passes
TAU_SAME          = 1.0      # PMI threshold for SAME
TAU_DIFF          = -0.5     # PMI threshold for DIFFERENT
BATCH_SIZE        = 512      # contrast batch size

print(f"Configuration:")
print(f"  V_MAX             = {V_MAX:,}")
print(f"  MIN_COUNT         = {MIN_COUNT}")
print(f"  WINDOW_SIZE       = {WINDOW_SIZE}")
print(f"  N_CONTRAST_PASSES = {N_CONTRAST_PASSES}")
print(f"  Total texts       = {len(all_texts):,}")

In [ ]:
# ── Create VocabularyBuilder ───────────────────────────────
builder = VocabularyBuilder(
    manifold=pipeline.manifold,
    annealing_engine=pipeline._annealing,
    contrast_engine=pipeline._contrast_engine,
    window_size=WINDOW_SIZE,
    min_count=MIN_COUNT,
    v_max=V_MAX,
    tau_same=TAU_SAME,
    tau_diff=TAU_DIFF,
    batch_size=BATCH_SIZE,
    n_contrast_passes=N_CONTRAST_PASSES,
)

print("✓ VocabularyBuilder created")

In [ ]:
# ── Feed all corpora ───────────────────────────────────────
# Shuffled to interleave domains — prevents the manifold from
# crystallising on one domain before seeing others.

import random
random.seed(42)
random.shuffle(all_texts)

t0 = time.time()
REPORT_EVERY = 2_500
n_total = len(all_texts)

for i, (text, source) in enumerate(all_texts):
    builder.feed(text)
    if (i + 1) % REPORT_EVERY == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        print(f"  [{i+1:>6,} / {n_total:,}]  "
              f"tokens: {builder.n_tokens_fed:>10,}  "
              f"({rate:.0f} texts/sec, {elapsed:.1f}s)")

elapsed_feed = time.time() - t0
print(f"\n✓ All corpora ingested")
print(f"  Texts processed : {n_total:,}")
print(f"  Total tokens    : {builder.n_tokens_fed:,}")
print(f"  Time            : {elapsed_feed:.1f}s")

## 5. Build Vocabulary — 6-step pipeline

Each step runs in its own cell with full progress logging:

1. **5A** — Build PMI matrix from co-occurrence counts
2. **5B** — Place vocabulary words on M(t) via C3 Annealing
3. **5C** — Contrast refinement via C4 (SAME/DIFFERENT from PMI)
4. **5D** — Calibrate phrase radius
5. **5E** — Build ExpressionEntry objects (Level 1 + 2 + 3)
6. **5F** — Save vocabulary to `.npz`

In [ ]:
# ── 5A: Build PMI matrix ───────────────────────────────────
import traceback

OUTPUT_DIR = "/kaggle/working"
GROUNDED_VOCAB_PATH    = f"{OUTPUT_DIR}/FLOW-Grounded_vocab.npz"
GROUNDED_MANIFOLD_PATH = f"{OUTPUT_DIR}/FLOW-Grounded_manifold.npz"

print("═" * 70)
print("  STEP 5A — Build PMI matrix")
print("═" * 70)
t0 = time.time()

try:
    matrix = builder._counter.build()
    builder._matrix = matrix
    elapsed_pmi = time.time() - t0

    print(f"  ✓ PMI matrix built in {elapsed_pmi:.1f}s")
    print(f"  Vocabulary size    : {len(matrix.vocabulary):,}")
    print(f"  PMI pairs          : {len(matrix._pmi):,}")
    print(f"  Directed PMI pairs : {len(matrix._dpmi):,}")
    print(f"  PMI max            : {matrix.pmi_max():.4f}")

    # Top-10 strongest PMI pairs
    top_pairs = matrix.pairs_above_threshold(1.0, -99)[:10]
    print(f"\n  Top 10 PMI pairs:")
    for w1, w2, pmi in top_pairs:
        print(f"    PMI={pmi:>7.3f}  {w1} ↔ {w2}")

    # Contrast workload preview
    same_pairs = [p for p in matrix.pairs_above_threshold(
        builder._scheduler.tau_same, -99) if p[2] > builder._scheduler.tau_same]
    diff_pairs = [p for p in matrix.pairs_above_threshold(
        99, builder._scheduler.tau_diff) if p[2] < builder._scheduler.tau_diff]
    causal_pairs = matrix.directed_pairs_above_delta(
        builder._scheduler.delta_causal)
    print(f"\n  Contrast workload preview:")
    print(f"    SAME pairs (PMI > {builder._scheduler.tau_same})  : {len(same_pairs):,}")
    print(f"    DIFF pairs (PMI < {builder._scheduler.tau_diff}) : {len(diff_pairs):,}")
    print(f"    Causal pairs (δ > {builder._scheduler.delta_causal})   : {len(causal_pairs):,}")
    est_total = (len(same_pairs) + len(diff_pairs)) * builder.n_contrast_passes
    print(f"    Est. total judgments                : ~{est_total:,}")

except Exception as e:
    elapsed_pmi = time.time() - t0
    print(f"  ✗ FAILED: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5A COMPLETE — {elapsed_pmi:.1f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5B: Place vocabulary words on M(t) ─────────────────────
from src.vocabulary.word_placer import WordPlacer, batch_structural_vectors_gpu

print("═" * 70)
print("  STEP 5B — Place vocabulary words on M(t)")
print("═" * 70)

vocab = matrix.vocabulary
freq_ranks = list(range(1, len(vocab) + 1))
n_words = len(vocab)
print(f"  Words to place     : {n_words:,}")
print(f"  Manifold before    : {pipeline.n_concepts:,} concepts")
print(f"  Temperature        : {pipeline.temperature:.6f}")

def placement_progress(i, total, label):
    elapsed = time.time() - t_place
    rate = i / elapsed if elapsed > 0 else 0
    pct = 100 * i / total
    print(f"    [{i:>6,}/{total:,}] {pct:5.1f}%  "
          f"{rate:,.0f} words/sec  "
          f"T={pipeline.temperature:.6f}  "
          f"last: {label}")

t_place = time.time()
try:
    try:
        import cupy as _cp
        print(f"  → Using GPU-accelerated structural vectors")
        placed = builder._placer.place_batch_gpu(vocab, freq_ranks,
                                                 progress_callback=placement_progress)
    except ImportError:
        print(f"  → Using optimized CPU batch placement")
        placed = builder._placer.place_batch(vocab, freq_ranks,
                                             progress_callback=placement_progress)

    builder._n_words_placed = len(vocab)
    elapsed_place = time.time() - t_place

    print(f"\n  ✓ Word placement complete")
    print(f"  Words placed       : {len(placed):,}")
    print(f"  Manifold after     : {pipeline.n_concepts:,} concepts")
    print(f"  Time               : {elapsed_place:.1f}s ({elapsed_place/60:.1f} min)")
    print(f"  Rate               : {len(placed)/elapsed_place:,.0f} words/sec")

    # Spot-check a few placements
    for w in vocab[:3]:
        lbl = f"vocab::{w}"
        try:
            pos = pipeline.manifold.position(lbl)
            d = pipeline.manifold.density(pos)
            print(f"    ✓ {lbl:30s}  ρ={d:.4f}  ‖P‖={np.linalg.norm(pos):.4f}")
        except (KeyError, ValueError) as e:
            print(f"    ✗ {lbl} — NOT FOUND: {e}")

except Exception as e:
    elapsed_place = time.time() - t_place
    print(f"\n  ✗ FAILED after {elapsed_place:.1f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5B COMPLETE — {elapsed_place:.1f}s ({elapsed_place/60:.1f} min)")
print(f"{'═' * 70}")

In [ ]:
# ── 5C: Contrast refinement — C4 SAME/DIFFERENT judgments ──────

print("═" * 70)
print("  STEP 5C — Contrast refinement (C4)")
print("═" * 70)
print(f"  Passes             : {builder.n_contrast_passes}")
print(f"  Batch size         : {builder._scheduler.batch_size}")
print(f"  τ_same             : {builder._scheduler.tau_same}")
print(f"  τ_diff             : {builder._scheduler.tau_diff}")
print(f"  Manifold labels    : {len(pipeline.manifold.labels):,}")

_contrast_last_report = [time.time()]
def contrast_progress(n_applied, n_total, phase):
    now = time.time()
    if now - _contrast_last_report[0] < 10.0 and "done" not in phase:
        return  # throttle to every 10s
    _contrast_last_report[0] = now
    elapsed = now - t_contrast
    if "pass" in phase and "done" in phase:
        print(f"    ── {phase}: {n_applied:,} judgments ({elapsed:.0f}s elapsed)")
    elif "done" in phase:
        print(f"    ── {phase}: {n_applied:,} judgments so far ({elapsed:.0f}s)")
    elif n_total > 0:
        pct = min(100, 100 * n_applied / max(n_total, 1))
        rate = n_applied / elapsed if elapsed > 0 else 0
        print(f"    [{n_applied:>8,} / ~{n_total:,}] {pct:5.1f}%  "
              f"{rate:,.0f} judg/sec ({elapsed:.0f}s)")

t_contrast = time.time()
try:
    n_judgments = builder._scheduler.run_passes(
        matrix,
        n_passes=builder.n_contrast_passes,
        progress_callback=contrast_progress,
    )
    builder._n_judgments = n_judgments
    elapsed_contrast = time.time() - t_contrast

    print(f"\n  ✓ Contrast refinement complete")
    print(f"  Total judgments    : {n_judgments:,}")
    print(f"  Time               : {elapsed_contrast:.1f}s ({elapsed_contrast/60:.1f} min)")
    print(f"  Rate               : {n_judgments/max(elapsed_contrast,0.1):,.0f} judgments/sec")

    # Spot-check geometry change
    if len(vocab) >= 2:
        w1, w2 = vocab[0], vocab[1]
        d = float(np.linalg.norm(
            pipeline.manifold.position(f"vocab::{w1}") -
            pipeline.manifold.position(f"vocab::{w2}")
        ))
        print(f"  Sample dist({w1}, {w2}) = {d:.4f}")

except Exception as e:
    elapsed_contrast = time.time() - t_contrast
    print(f"\n  ✗ FAILED after {elapsed_contrast:.1f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5C COMPLETE — {elapsed_contrast:.1f}s ({elapsed_contrast/60:.1f} min)")
print(f"{'═' * 70}")

In [ ]:
# ── 5D: Calibrate phrase radius ─────────────────────────────
from src.vocabulary.template_builder import TemplateBuilder

print("═" * 70)
print("  STEP 5D — Calibrate phrase radius")
print("═" * 70)

t_calib = time.time()
try:
    tb = TemplateBuilder(pipeline.manifold)
    radius = tb.calibrate_phrase_radius()
    elapsed_calib = time.time() - t_calib

    print(f"  ✓ Phrase radius calibrated")
    print(f"  Radius             : {radius:.4f}")
    print(f"  Time               : {elapsed_calib:.2f}s")

    vocab_labels = [l for l in pipeline.manifold._points if l.startswith("vocab::")]
    print(f"  Vocab on manifold  : {len(vocab_labels):,}")

except Exception as e:
    elapsed_calib = time.time() - t_calib
    print(f"\n  ✗ FAILED after {elapsed_calib:.2f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5D COMPLETE — {elapsed_calib:.2f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5E: Build ExpressionEntry objects (Level 1 + 2 + 3) ────────

print("═" * 70)
print("  STEP 5E — Build ExpressionEntry objects")
print("═" * 70)

t_build = time.time()
try:
    # Level 1 — single words
    print("  Building Level 1 (single words)...")
    t_l1 = time.time()
    l1_entries = tb._build_level1()
    elapsed_l1 = time.time() - t_l1
    print(f"    ✓ Level 1: {len(l1_entries):,} entries in {elapsed_l1:.1f}s")
    if l1_entries:
        s = l1_entries[0]
        print(f"    Sample: '{s.text}' register={s.register} "
              f"causal={s.causal_strength:.3f} hedging={s.hedging}")

    # Level 2 — short phrases
    print("  Building Level 2 (short phrases)...")
    t_l2 = time.time()
    l2_entries = tb._build_level2(matrix)
    elapsed_l2 = time.time() - t_l2
    print(f"    ✓ Level 2: {len(l2_entries):,} entries in {elapsed_l2:.1f}s")
    if l2_entries:
        s = l2_entries[0]
        print(f"    Sample: '{s.text}' register={s.register}")

    # Level 3 — sentence frames
    print("  Building Level 3 (sentence frames)...")
    t_l3 = time.time()
    l3_entries = tb._build_level3()
    elapsed_l3 = time.time() - t_l3
    print(f"    ✓ Level 3: {len(l3_entries):,} entries in {elapsed_l3:.1f}s")
    if l3_entries:
        s = l3_entries[0]
        print(f"    Sample: '{s.text}'")

    # Deduplicate
    new_entries = []
    seen_texts = set()
    for e in l1_entries + l2_entries + l3_entries:
        if e.text not in seen_texts:
            seen_texts.add(e.text)
            new_entries.append(e)

    elapsed_build = time.time() - t_build

    print(f"\n  ✓ All entries built and deduplicated")
    print(f"  Level 1            : {len(l1_entries):,}  ({elapsed_l1:.1f}s)")
    print(f"  Level 2            : {len(l2_entries):,}  ({elapsed_l2:.1f}s)")
    print(f"  Level 3            : {len(l3_entries):,}  ({elapsed_l3:.1f}s)")
    print(f"  New entries (dedup): {len(new_entries):,}")
    print(f"  Total time         : {elapsed_build:.1f}s ({elapsed_build/60:.1f} min)")

    # Register distribution
    reg_dist = Counter(e.register for e in new_entries)
    hedging_count = sum(1 for e in new_entries if e.hedging)
    print(f"\n  Register distribution:")
    for reg, cnt in reg_dist.most_common():
        print(f"    {reg:10s} : {cnt:>6,}")
    print(f"  Hedging entries    : {hedging_count:,}")

except Exception as e:
    elapsed_build = time.time() - t_build
    print(f"\n  ✗ FAILED after {elapsed_build:.1f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5E COMPLETE — {elapsed_build:.1f}s ({elapsed_build/60:.1f} min)")
print(f"{'═' * 70}")

In [ ]:
# ── 5F: Merge with base vocabulary & save ───────────────────
# Combine the K-12 base vocabulary with new PMI-grounded entries.
# Deduplicate by text — if a word appears in both, keep the new
# PMI-grounded version (better wave_profile from real co-occurrence).

print("═" * 70)
print("  STEP 5F — Merge & save vocabulary")
print("═" * 70)

t_save = time.time()
try:
    # Merge: new entries take priority over base entries
    new_texts = {e.text for e in new_entries}
    base_kept = [e for e in base_vocab_entries if e.text not in new_texts]
    merged_entries = new_entries + base_kept

    print(f"  New PMI entries    : {len(new_entries):,}")
    print(f"  Base entries kept  : {len(base_kept):,} (of {n_base_vocab:,})")
    print(f"  Base entries replaced: {n_base_vocab - len(base_kept):,}")
    print(f"  Merged total       : {len(merged_entries):,}")

    if not merged_entries:
        raise RuntimeError("No entries to save — check steps 5A–5E.")

    n_entries = VocabularyStore.save(merged_entries, GROUNDED_VOCAB_PATH)
    elapsed_save = time.time() - t_save

    print(f"\n  ✓ Vocabulary saved")
    print(f"  Entries written    : {n_entries:,}")
    print(f"  File               : {GROUNDED_VOCAB_PATH}")
    print(f"  File size          : {os.path.getsize(GROUNDED_VOCAB_PATH) / 1024 / 1024:.2f} MB")
    print(f"  Time               : {elapsed_save:.2f}s")

except Exception as e:
    elapsed_save = time.time() - t_save
    print(f"\n  ✗ FAILED after {elapsed_save:.2f}s: {e}")
    traceback.print_exc()
    raise

# ── Full pipeline timing summary ────────────────────────────
total_build_time = elapsed_pmi + elapsed_place + elapsed_contrast + elapsed_calib + elapsed_build + elapsed_save
print(f"\n{'═' * 70}")
print(f"  5F COMPLETE — vocabulary saved")
print(f"{'═' * 70}")
print(f"\n  ┌─────────────────────────────────────────────────┐")
print(f"  │  BUILD TIMING SUMMARY                           │")
print(f"  ├─────────────────────────────────────────────────┤")
print(f"  │  5A PMI matrix      : {elapsed_pmi:>8.1f}s  ({elapsed_pmi/60:>5.1f} min)  │")
print(f"  │  5B Word placement  : {elapsed_place:>8.1f}s  ({elapsed_place/60:>5.1f} min)  │")
print(f"  │  5C Contrast passes : {elapsed_contrast:>8.1f}s  ({elapsed_contrast/60:>5.1f} min)  │")
print(f"  │  5D Phrase radius   : {elapsed_calib:>8.1f}s  ({elapsed_calib/60:>5.1f} min)  │")
print(f"  │  5E Build entries   : {elapsed_build:>8.1f}s  ({elapsed_build/60:>5.1f} min)  │")
print(f"  │  5F Save .npz       : {elapsed_save:>8.1f}s  ({elapsed_save/60:>5.1f} min)  │")
print(f"  ├─────────────────────────────────────────────────┤")
print(f"  │  TOTAL              : {total_build_time:>8.1f}s  ({total_build_time/60:>5.1f} min)  │")
print(f"  └─────────────────────────────────────────────────┘")

In [ ]:
# ── Save manifold state ───────────────────────────────────

n_saved = ManifoldSnapshot.save(pipeline.manifold, GROUNDED_MANIFOLD_PATH)

print(f"✓ Manifold state saved")
print(f"  Concepts on M(t)   : {n_saved:,}")
print(f"  File               : {GROUNDED_MANIFOLD_PATH}")
print(f"  File size          : {os.path.getsize(GROUNDED_MANIFOLD_PATH) / 1024 / 1024:.2f} MB")
print(f"  Temperature        : {pipeline.temperature:.6f}")

## 6. Verify Artifacts

In [ ]:
# ── Verify manifold round-trip ─────────────────────────────
info = ManifoldSnapshot.info(GROUNDED_MANIFOLD_PATH)
print("Manifold snapshot info:")
for k, v in info.items():
    print(f"  {k:20s} : {v}")

# Spot-check: reload and compare positions
reloaded = ManifoldSnapshot.load(GROUNDED_MANIFOLD_PATH)
labels = list(pipeline.manifold._points.keys())
max_err = 0.0
for label in labels[:100]:
    orig = pipeline.manifold.position(label)
    rest = reloaded.position(label)
    err = np.max(np.abs(orig - rest))
    max_err = max(max_err, err)

print(f"\n✓ Manifold round-trip verified")
print(f"  Max position error : {max_err:.2e}")
assert max_err < 1e-10, f"Round-trip error too large: {max_err}"

# Verify vocabulary
loaded_entries = VocabularyStore.load(GROUNDED_VOCAB_PATH)
print(f"\nVocabulary verification:")
print(f"  Entries loaded     : {len(loaded_entries):,}")
print(f"\n  Sample entries (first 20):")
for entry in loaded_entries[:20]:
    print(f"    [{entry.register:>7s}] [{entry.rhythm:>6s}] {entry.text[:60]}")

## 7. Evaluate — Query the Grounded Manifold

In [ ]:
# ── Load grounded vocabulary into the renderer ───────────────
# Use extend + _faiss_dirty to load in-memory entries
pipeline._renderer.matcher.vocabulary.clear()
pipeline._renderer.matcher.vocabulary.extend(loaded_entries)
pipeline._renderer.matcher._faiss_dirty = True
print(f"✓ Loaded {len(loaded_entries):,} entries into C7 renderer")

# ── Sample queries ───────────────────────────────────────
all_labels = list(pipeline.manifold._points.keys())
vocab_labels = [l for l in all_labels if l.startswith("vocab::")]
print(f"Vocabulary concepts on manifold: {len(vocab_labels):,}")
print(f"Total concepts on manifold     : {len(all_labels):,}")

sample_words = vocab_labels[:10] if len(vocab_labels) >= 10 else vocab_labels
print(f"\nSample queries:")
print("=" * 70)

for label in sample_words:
    vec = pipeline.manifold.position(label)
    result = pipeline.query(vec, label=label)
    word = label.replace("vocab::", "")
    print(f"\n  Query: {word}")
    print(f"  Steps: {result.n_steps}  |  Reason: {result.termination_reason}")
    print(f"  Confidence: {result.confidence:.3f}  |  Wave: {result.wave_confidence:.3f}")
    print(f"  Output: {result.text[:120]}")
    print("-" * 70)

In [ ]:
# ── Evaluation suite ─────────────────────────────────────
from src.phase5 import PipelineEvaluator

evaluator = PipelineEvaluator(pipeline)

n_eval = min(50, len(vocab_labels))
eval_labels = vocab_labels[:n_eval]
eval_vecs = [pipeline.manifold.position(l) for l in eval_labels]

print(f"Running evaluation suite with {n_eval} queries...")
t0 = time.time()
suite = evaluator.run_suite(eval_vecs, eval_labels)
elapsed_eval = time.time() - t0

print(f"\n✓ Evaluation complete ({elapsed_eval:.1f}s)")
print(f"  Mean coherence       : {suite.mean_coherence:.4f}")
print(f"  Novelty is decaying  : {suite.novelty_is_decaying}")
print(f"  Termination reasons  :")
for reason, count in suite.termination_distribution.items():
    print(f"    {reason:30s} : {count}")

## 8. Summary

In [ ]:
# ── Final summary ─────────────────────────────────────────
print("=" * 70)
print("  FLOW — Vocabulary Growth (Corpus-Grounded) — Final Report")
print("=" * 70)
print(f"")
print(f"  Base model: {base_name}")
print(f"")
print(f"  Corpora")
print(f"    Wikipedia          : {source_counts.get('wikipedia', 0):,}")
print(f"    OpenAssistant      : {source_counts.get('openassistant', 0):,}")
print(f"    Alpaca (cleaned)   : {source_counts.get('alpaca', 0):,}")
print(f"    Glaive Functions   : {source_counts.get('glaive_functions', 0):,}")
print(f"    SlimOrca           : {source_counts.get('slimorca', 0):,}")
print(f"    Total texts        : {len(all_texts):,}")
print(f"    Total tokens       : {builder.n_tokens_fed:,}")
print(f"")
print(f"  Vocabulary")
print(f"    New PMI entries     : {len(new_entries):,}")
print(f"    Base entries kept   : {len(base_kept):,}")
print(f"    Merged total        : {n_entries:,}")
print(f"    Contrast judgments  : {builder.n_judgments_applied:,}")
print(f"    vocab.npz size     : {os.path.getsize(GROUNDED_VOCAB_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Manifold")
print(f"    Total concepts     : {pipeline.n_concepts:,}")
print(f"    Dimension          : {pipeline.manifold.DIM}")
print(f"    Temperature        : {pipeline.temperature:.6f}")
print(f"    manifold.npz size  : {os.path.getsize(GROUNDED_MANIFOLD_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Evaluation")
print(f"    Mean coherence     : {suite.mean_coherence:.4f}")
print(f"    Novelty decaying   : {suite.novelty_is_decaying}")
print(f"")
print(f"  Build time           : {total_build_time:.0f}s ({total_build_time/60:.1f} min)")
print(f"  Corpus feed time     : {elapsed_feed:.0f}s ({elapsed_feed/60:.1f} min)")
print("=" * 70)

## 9. Save & Push to GitHub

In [ ]:
# ── Push grounded model to GitHub ────────────────────────────
import shutil

if PAT:
    models_dir = f"{REPO_DIR}/models"
    os.makedirs(models_dir, exist_ok=True)

    # Copy artifacts into repo
    shutil.copy2(GROUNDED_MANIFOLD_PATH, f"{models_dir}/FLOW-Grounded_manifold.npz")
    shutil.copy2(GROUNDED_VOCAB_PATH,    f"{models_dir}/FLOW-Grounded_vocab.npz")
    print(f"✓ Artifacts copied to {models_dir}/")

    # Configure git
    subprocess.run(["git", "config", "user.email", "kaggle@flow.dev"],
                   cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "config", "user.name", "FLOW-Kaggle"],
                   cwd=REPO_DIR, capture_output=True)

    # Stage, commit, push
    cmds = [
        ["git", "add",
         "models/FLOW-Grounded_manifold.npz",
         "models/FLOW-Grounded_vocab.npz"],
        ["git", "commit", "-m",
         f"Grounded vocab: {n_entries:,} entries, "
         f"{pipeline.n_concepts:,} concepts, "
         f"{builder.n_tokens_fed:,} tokens from 5 corpora"],
        ["git", "push", "origin", "main"],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        if r.returncode != 0 and "nothing to commit" not in r.stdout:
            print(f"  ✘ {' '.join(cmd[:3])}: {r.stderr.strip()[:120]}")

    final_hash = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    print(f"✓ Pushed to GitHub (commit {final_hash})")
    print(f"  FLOW-Grounded_manifold.npz")
    print(f"  FLOW-Grounded_vocab.npz")
else:
    print("⚠ No PAT — artifacts saved to /kaggle/working/ only")
    print(f"  {GROUNDED_MANIFOLD_PATH}")
    print(f"  {GROUNDED_VOCAB_PATH}")

print(f"\nTo reload locally:")
print(f"  pipeline = GEOPipeline.load(")
print(f"      'FLOW-Grounded_manifold.npz',")
print(f"      vocabulary_path='FLOW-Grounded_vocab.npz')")